<a href="https://colab.research.google.com/github/LEIXINREN/TheLongDark_Save_Unpack/blob/main/the_long_dark_save_to_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-lzf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for python-lzf: filename=python_lzf-0.2.6-cp313-cp313-linux_x86_64.whl size=18850 sha256=8610b7b734caded094b769c5fdf2ef24cfec50abea5c9dabe5b855a4ac146c08
  Stored in directory: /root/.cache/pip/wheels/96/99/fd/42828889a28f00575d52f960badb0e56b1e38d862277a2d6cf
Successfully built python-lzf


# Base64 Encoding

In [ ]:
import base64
modified_file = "sandbox1"
output_b64_file = "base64_save.txt"

with open(modified_file, "rb") as f:
    binary_data = f.read()

encoded_data = base64.b64encode(binary_data).decode("utf-8")

with open(output_b64_file, "w", encoding="utf-8") as f:
    f.write(encoded_data)

print(f"{output_b64_file}")

base64_save.txt


# LZF Decoding

In [ ]:
import base64
import lzf

input_file = "base64_save.txt"
output_file = "lzf_base64_save.txt"

with open(input_file, "rb") as f:
    binary_content = f.read()

encoded_data = binary_content.decode("utf-8", errors="ignore").strip()
decoded_bytes = base64.b64decode(encoded_data)
print(f"Base64 decoding completed. Size: {len(decoded_bytes)} bytes")

# Automatically guess decompressed size by starting at 2x and scaling up
decompressed_bytes = None
original_size = len(decoded_bytes) * 2  # Initial guess size

for i in range(10):  # Maximum of 10 attempts
    try:
        # lzf.decompress requires: (data, expected_decompressed_size)
        decompressed_bytes = lzf.decompress(decoded_bytes, original_size)
        print(f"Decompression successful using buffer size: {original_size} bytes")
        break
    except Exception as e:
        # If output buffer is too small, double the size and try again
        original_size *= 2
        if i == 9:
            print(f"Decompression failed. Final error: {e}")

if decompressed_bytes:
    with open(output_file, "wb") as f:
        f.write(decompressed_bytes)
    print(f"Successfully saved to {output_file}. Final size: {len(decompressed_bytes)} bytes")
else:
    print("Could not complete decompression. Verify if the file is indeed LZF-compressed.")

Base64 decoding completed. Size: 11007215 bytes
Decompression successful using buffer size: 22014430 bytes
Successfully saved to lzf_base64_save.txt. Final size: 11433187 bytes


# Decoding


In [ ]:
import base64
import json
from google.colab import files

def lzf_decompress(compressed_data: bytes) -> bytes:
    i = 0
    out = bytearray()
    data_len = len(compressed_data)

    while i < data_len:
        ctrl = compressed_data[i]
        i += 1

        if ctrl < 32:
            length = ctrl + 1
            out.extend(compressed_data[i : i + length])
            i += length
        else:
            length = ctrl >> 5
            ref_offset = ((ctrl & 0x1F) << 8) + 1

            if length == 7:
                length += compressed_data[i]
                i += 1

            ref_offset += compressed_data[i]
            i += 1
            length += 2

            ref_pos = len(out) - ref_offset
            for _ in range(length):
                out.append(out[ref_pos])
                ref_pos += 1

    return bytes(out)

input_file = "lzf_base64_save.txt"
print(f" Processing static input file: {input_file}")

# ==========================================
# SetArea
# ==========================================
#target_keys = ["global", "boot","RuralRegion"]
target_keys = None              # all

try:
    with open(input_file, 'rb') as f:
        raw_data = f.read()

    try:
        file_content = raw_data.decode('utf-8').strip()
        save_data = json.loads(file_content)

        if "m_Dict" in save_data:
            print(" Detected full save structure with m_Dict.")
            decoded_dict = {}

            for key, b64_string in save_data["m_Dict"].items():

                if target_keys is not None and key not in target_keys:
                    continue

                print(f"\n--- Decoding: m_Dict['{key}'] ---")
                try:
                    # --- Base64 Fix ---
                    padding_needed = (4 - len(b64_string) % 4) % 4
                    if padding_needed:
                        b64_string += "=" * padding_needed

                    compressed_bytes = base64.b64decode(b64_string)
                    json_bytes = lzf_decompress(compressed_bytes)
                    decoded_text = json_bytes.decode('utf-8', errors='replace')

                    try:
                        decoded_dict[key] = json.loads(decoded_text)
                    except json.JSONDecodeError:
                        decoded_dict[key] = decoded_text


                    ind_filename = f"decoded_{key}.json"
                    with open(ind_filename, 'w', encoding='utf-8') as ind_f:
                        try:
                            parsed_json = json.loads(decoded_text)
                            json.dump(parsed_json, ind_f, ensure_ascii=False, indent=4)
                        except:
                            ind_f.write(decoded_text)
                    print(f" Successfully saved files: {ind_filename}")

                except Exception as ex:
                    print(f" Processing '{key}' error: {ex}")
                    decoded_dict[key] = f"Error: {ex}"


            output_filename = "decoded_save_selected.json"
            with open(output_filename, 'w', encoding='utf-8') as f:
                json.dump(decoded_dict, f, ensure_ascii=False, indent=4)
            print(f"\n The specified fields have been processed! The file has been saved. {output_filename}")

        else:
            print(" No m_Dict detected, treating the whole file as base64.")
            b64_string = file_content
            padding_needed = (4 - len(b64_string) % 4) % 4
            if padding_needed:
                b64_string += "=" * padding_needed

            compressed_bytes = base64.b64decode(b64_string)
            json_bytes = lzf_decompress(compressed_bytes)
            final_json = json_bytes.decode('utf-8', errors='replace')

            output_filename = "decoded_save.json"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(final_json)
            print(f" Success! Saved as {output_filename}")

    except UnicodeDecodeError:
        print(" Binary data detected, treating raw data directly as compressed bytes.")
        compressed_bytes = raw_data
        json_bytes = lzf_decompress(compressed_bytes)
        final_json = json_bytes.decode('utf-8', errors='replace')

        output_filename = "decoded_save.json"
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(final_json)
        print(f" Success! Saved as {output_filename}")

except Exception as e:
    print(f" Error processing file: {e}")

# \\" to "

In [ ]:
file_path = 'decoded_RiverValleyRegion.json'


with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()


new_content = content.replace(r'\"', '"')


with open(file_path, 'w', encoding='utf-8') as f:
    f.write(new_content)


Norton_ 20261119